In [ ]:
# ============================================================
# QUESTION 4 - SETUP
# ============================================================

import nltk
import random
import time
import math
import os
import pickle
import statistics
from collections import Counter, defaultdict

import pandas as pd

from nltk.corpus import treebank, gutenberg, brown
from nltk import induce_pcfg
from nltk import Tree

# Required corpora
nltk.download("treebank")
nltk.download("gutenberg")
nltk.download("punkt")
nltk.download("punkt_tab")

print("Q4 setup complete.")

Q4 setup complete.


[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Package treebank is already up-to-date!
[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
from tqdm.notebook import tqdm

In [ ]:
# ============================================================
# LOAD Q1 + Q3 TRAINED COMPONENTS
# ============================================================

from google.colab import drive
import os
import cloudpickle

drive.mount("/content/drive")

MODEL_DIR = "/content/drive/MyDrive/GROUP_ASSIGNMENT_1/models"

Q1_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "q1_models.pkl"
)

Q3_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "q3_models.pkl"
)

# ------------------------------------------------------------
# Load Q1
# ------------------------------------------------------------

with open(Q1_MODEL_PATH, "rb") as f:
    q1_models = cloudpickle.load(f)

lm_en = q1_models["lm_en"]
hmm_en = q1_models["hmm_en"]
vocab_en = q1_models["vocab_en"]

# ------------------------------------------------------------
# Load Q3
# ------------------------------------------------------------

with open(Q3_MODEL_PATH, "rb") as f:
    q3_models = cloudpickle.load(f)

vocab = q3_models["vocab"]
unigram_freq = q3_models["unigram_freq"]
bigram_freq = q3_models["bigram_freq"]
unigram_ctx_freq = q3_models["unigram_ctx_freq"]
symspell_index = q3_models["symspell_index"]

correct_nonword = q3_models["correct_nonword"]
correct_realword = q3_models["correct_realword"]

print("✓ Q1 models loaded")
print("✓ Q3 models loaded")
print("✓ Q4 will reuse Q1 and Q3 components")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Q1 models loaded
✓ Q3 models loaded
✓ Q4 will reuse Q1 and Q3 components


In [ ]:
# ============================================================
# Q4 CONFIGURATION
# ============================================================

MERGE_PROBABILITY = 0.08
GRAMMAR_TRIGGER_N = 8

# Q1's already-selected settings
Q1_MAX_WORD_LEN = lm_en.max_word_len
Q1_BEAM_SIZE = 25

print("Q4 configuration")
print("-----------------")
print(f"Merge probability p       = {MERGE_PROBABILITY}")
print(f"Grammar trigger interval N = {GRAMMAR_TRIGGER_N}")
print(f"Q1 max word length        = {Q1_MAX_WORD_LEN}")
print(f"Q1 beam width             = {Q1_BEAM_SIZE}")

print("\nJustification:")
print("p = 0.08 gives realistic occasional fast-typing space omissions.")
print("N = 8 balances responsiveness with the cost of contextual grammar checks.")

Q4 configuration
-----------------
Merge probability p       = 0.08
Grammar trigger interval N = 8
Q1 max word length        = 18
Q1 beam width             = 25

Justification:
p = 0.08 gives realistic occasional fast-typing space omissions.
N = 8 balances responsiveness with the cost of contextual grammar checks.


In [ ]:
# ============================================================
# PART 2 - PCFG TRAINING
# ============================================================

print("Loading Penn Treebank...")

treebank_trees = treebank.parsed_sents()

# Use the complete Penn Treebank for better lexical coverage.
PCFG_TRAIN_SIZE = len(treebank_trees)

pcfg_trees = []

for original_tree in treebank_trees[:PCFG_TRAIN_SIZE]:

    # Work on a copy so the original Treebank tree is unchanged.
    tree = original_tree.copy(deep=True)

    # Lowercase terminal words.
    for pos in tree.treepositions("leaves"):
        if isinstance(tree[pos], str):
            tree[pos] = tree[pos].lower()

    # Convert the tree into CNF form for CKY parsing.
    tree.chomsky_normal_form(horzMarkov=0)

    pcfg_trees.append(tree)

if not pcfg_trees:
    raise RuntimeError(
        "Penn Treebank is empty; cannot train the PCFG."
    )

# Collect all grammar productions.
productions = [
    prod
    for tree in pcfg_trees
    for prod in tree.productions()
]

# NLTK induce_pcfg() requires a Nonterminal object.
from nltk.grammar import Nonterminal

pcfg_start_symbol = Nonterminal(
    pcfg_trees[0].label()
)

# Induce the probabilistic context-free grammar.
pcfg_grammar = induce_pcfg(
    pcfg_start_symbol,
    productions
)

print(
    f"Treebank trees used: {PCFG_TRAIN_SIZE}"
)

print(
    f"Number of PCFG productions: "
    f"{len(pcfg_grammar.productions())}"
)

print(
    f"PCFG start symbol: "
    f"{pcfg_grammar.start().symbol()}"
)

print("PCFG training complete.")

Loading Penn Treebank...
Treebank trees used: 3914
Number of PCFG productions: 18998
PCFG start symbol: S
PCFG training complete.


In [ ]:
# ============================================================
# PCFG INDEXES
# ============================================================

grammar_productions = pcfg_grammar.productions()

# Binary productions: A -> B C
binary_rules = defaultdict(list)

# Unary non-terminal productions: A -> B
unary_rules = defaultdict(list)

# Lexical productions: A -> 'word'
lexical_rules = defaultdict(list)

nonterminals = set()

for prod in grammar_productions:
    lhs = prod.lhs()
    rhs = prod.rhs()

    nonterminals.add(lhs)

    if len(rhs) == 2 and all(hasattr(x, "symbol") for x in rhs):
        binary_rules[(rhs[0], rhs[1])].append(
            (lhs, prod.prob())
        )

    elif len(rhs) == 1 and hasattr(rhs[0], "symbol"):
        unary_rules[rhs[0]].append(
            (lhs, prod.prob())
        )

    elif len(rhs) == 1:
        lexical_rules[str(rhs[0]).lower()].append(
            (lhs, prod.prob())
        )

print(f"Non-terminals: {len(nonterminals)}")
print(f"Binary rule groups: {len(binary_rules)}")
print(f"Unary rule groups: {len(unary_rules)}")
print(f"Known lexical items: {len(lexical_rules)}")

Non-terminals: 899
Binary rule groups: 3041
Unary rule groups: 45
Known lexical items: 11387


In [ ]:
# ============================================================
# TAGSET RECONCILIATION
# ============================================================

Q1_TO_PTB = {
    "NOUN": "NN",
    "PROPN": "NNP",
    "VERB": "VB",
    "AUX": "VB",
    "ADJ": "JJ",
    "ADV": "RB",
    "PRON": "PRP",
    "DET": "DT",
    "ADP": "IN",
    "NUM": "CD",
    "CONJ": "CC",
    "CCONJ": "CC",
    "SCONJ": "IN",
    "PART": "RP",
    "INTJ": "UH",
    "SYM": "SYM",
    "X": "FW"
}


def reconcile_q1_tags(words):

    words = [
        w.lower()
        for w in words
    ]

    alpha_words = [
        w
        for w in words
        if w.isalpha()
    ]

    alpha_tags = hmm_en.tag(
        alpha_words
    )

    q1_tags = []
    ptb_tags = []

    j = 0

    for word in words:

        if word.isalpha():

            tag = alpha_tags[j]
            j += 1

            q1_tags.append(tag)
            ptb_tags.append(
                Q1_TO_PTB.get(tag)
            )

        else:

            q1_tags.append(None)
            ptb_tags.append(None)

    return q1_tags, ptb_tags

In [ ]:
# ============================================================
# CKY / VITERBI PARSER
# ============================================================

def safe_log(prob):
    if prob <= 0:
        return -1e9
    return math.log(prob)


# Small penalty used when a word is not present in the
# Treebank lexical vocabulary but Q1 provides a plausible
# PTB preterminal tag.
UNKNOWN_WORD_PENALTY = -20.0


def apply_unary_closure(chart, i, j):
    """
    Add unary non-terminal expansions until no new
    higher-scoring constituent can be added.
    """
    changed = True

    while changed:
        changed = False

        current_items = list(
            chart[i][j].items()
        )

        for child, (score, tree) in current_items:

            for parent, prob in unary_rules.get(
                child,
                []
            ):

                new_score = (
                    score
                    + safe_log(prob)
                )

                if (
                    parent not in chart[i][j]
                    or new_score > chart[i][j][parent][0]
                ):

                    chart[i][j][parent] = (
                        new_score,
                        Tree(
                            parent.symbol(),
                            [tree]
                        )
                    )

                    changed = True


def cky_parse(
    words,
    q1_ptb_tags=None
):
    """
    Viterbi/CKY most-probable constituency parse.

    Q1 POS predictions are used as a soft lexical constraint.
    For words missing from the PCFG lexical vocabulary, a
    Q1-compatible PTB preterminal is used as an unknown-word
    lexical backoff with a penalty.

    A successful parse must span the complete sentence and
    must use the PCFG start symbol.
    """

    words = [
        str(w).lower()
        for w in words
        if str(w).strip()
    ]

    n = len(words)

    if n == 0:
        return None, -1e9

    chart = [
        [
            {}
            for _ in range(n + 1)
        ]
        for _ in range(n)
    ]

    # --------------------------------------------------------
    # Initialization: lexical productions
    # --------------------------------------------------------
    for i, word in enumerate(words):

        lexical_entries = list(
            lexical_rules.get(
                word,
                []
            )
        )

        preferred_tag = None

        if (
            q1_ptb_tags is not None
            and i < len(q1_ptb_tags)
        ):
            preferred_tag = q1_ptb_tags[i]

        # ----------------------------------------------------
        # Known word
        # ----        if lexical_entries:

            entries_to_use = list(lexical_entries)

            if preferred_tag is not None:

                from nltk.grammar import Nonterminal

                preferred_symbol = Nonterminal(preferred_tag)

                has_preferred = any(
                    item[0] == preferred_symbol
                    for item in entries_to_use
                )

                # Q1's tag isn't among this word's WSJ entries —
                # add it as a low-confidence extra option instead
                # of only accepting whatever tag WSJ happened to
                # give the word (e.g. "cat"/"sat" only ever show
                # up as NNP in the WSJ sample).
                if (
                    not has_preferred
                    and preferred_symbol in nonterminals
                ):
                    entries_to_use.append(
                        (preferred_symbol, None)
                    )

            for parent, prob in entries_to_use:

                score = (
                    UNKNOWN_WORD_PENALTY
                    if prob is None
                    else safe_log(prob)
                )

                leaf_tree = Tree(
                    parent.symbol(),
                    [word]
                )

                if (
                    parent not in chart[i][i + 1]
                    or
                    score
                    > chart[i][i + 1][parent][0]
                ):

                    chart[i][i + 1][parent] = (
                        score,
                        leaf_tree
                    )

        # ----------------------------------------------------
        # Unknown word
        # ----------------------------------------------------
        else:

            if preferred_tag is not None:

                from nltk.grammar import Nonterminal

                preferred_symbol = Nonterminal(
                    preferred_tag
                )

                # Use the Q1-compatible PTB tag only when it
                # is actually a known non-terminal in the grammar.
                if preferred_symbol in nonterminals:

                    chart[i][i + 1][
                        preferred_symbol
                    ] = (
                        UNKNOWN_WORD_PENALTY,
                        Tree(
                            preferred_tag,
                            [word]
                        )
                    )

        # If neither normal lexical entries nor a valid
        # unknown-word backoff exists, the sentence cannot
        # be parsed.
        if not chart[i][i + 1]:
            return None, -1e9

        apply_unary_closure(
            chart,
            i,
            i + 1
        )

    # --------------------------------------------------------
    # CKY dynamic programming
    # --------------------------------------------------------
    for span in range(
        2,
        n + 1
    ):

        for i in range(
            0,
            n - span + 1
        ):

            j = i + span

            for k in range(
                i + 1,
                j
            ):

                left_items = list(
                    chart[i][k].items()
                )

                right_items = list(
                    chart[k][j].items()
                )

                for (
                    left_symbol,
                    (
                        left_score,
                        left_tree
                    )
                ) in left_items:

                    for (
                        right_symbol,
                        (
                            right_score,
                            right_tree
                        )
                    ) in right_items:

                        candidates = binary_rules.get(
                            (
                                left_symbol,
                                right_symbol
                            ),
                            []
                        )

                        for parent, prob in candidates:

                            score = (
                                left_score
                                + right_score
                                + safe_log(prob)
                            )

                            if (
                                parent not in chart[i][j]
                                or
                                score
                                > chart[i][j][parent][0]
                            ):

                                chart[i][j][parent] = (
                                    score,
                                    Tree(
                                        parent.symbol(),
                                        [
                                            left_tree,
                                            right_tree
                                        ]
                                    )
                                )

            apply_unary_closure(
                chart,
                i,
                j
            )

    # --------------------------------------------------------
    # Final result
    # --------------------------------------------------------
    root = pcfg_grammar.start()

    if root in chart[0][n]:

        return (
            chart[0][n][root][1],
            chart[0][n][root][0]
        )

    # Do not accept a partial constituent as a sentence parse.
    return None, -1e9

In [ ]:
# ============================================================
# TEST CKY PARSER
# ============================================================

test_sentences = [
    "the cat sat on the mat",
    "she eats a green salad",
    "i saw the man with a telescope"
]

for sentence in test_sentences:

    words = sentence.split()

    q1_tags, ptb_tags = reconcile_q1_tags(words)

    tree, score = cky_parse(
        words,
        q1_ptb_tags=ptb_tags
    )

    print("\nSentence:", sentence)

    if tree is None:
        print("UNPARSEABLE")
    else:
        print("Parse score:", score)
        print(tree)


Sentence: the cat sat on the mat
Parse score: -56.22354923085897
(S
  (NP-SBJ (DT the) (NP-SBJ|<> (NNP cat) (NNP sat)))
  (PP-PRD (IN on) (NP (DT the) (NN mat))))

Sentence: she eats a green salad
Parse score: -61.35170296460404
(S
  (NP-SBJ (PRP she))
  (VP (VB eats) (NP (DT a) (NP|<> (JJ green) (NN salad)))))

Sentence: i saw the man with a telescope
Parse score: -54.81465900714462
(S
  (NP-SBJ (PRP i))
  (VP
    (VBD saw)
    (VP|<>
      (NP (DT the) (NN man))
      (PP-CLR (IN with) (NP (DT a) (NN telescope))))))


In [ ]:
# ============================================================
# PCFG LEXICAL COVERAGE CHECK
# ============================================================

check_words = [
    "the",
    "cat",
    "sat",
    "on",
    "mat",
    "she",
    "eats",
    "a",
    "green",
    "salad",
    "i",
    "saw",
    "man",
    "with",
    "telescope"
]

print("PCFG start symbol:", pcfg_grammar.start())
print("Number of grammar productions:", len(pcfg_grammar.productions()))
print("Number of lexical items:", len(lexical_rules))
print()

for word in check_words:

    entries = lexical_rules.get(word, [])

    if entries:
        tags = sorted(
            set(parent.symbol() for parent, prob in entries)
        )
        print(f"{word:12s} -> FOUND    {tags}")
    else:
        print(f"{word:12s} -> MISSING")

PCFG start symbol: S
Number of grammar productions: 18998
Number of lexical items: 11387

the          -> FOUND    ['CD', 'DT', 'JJ', 'NNP']
cat          -> FOUND    ['NNP']
sat          -> FOUND    ['NNP']
on           -> FOUND    ['IN', 'NNP', 'RB', 'RP']
mat          -> MISSING
she          -> FOUND    ['PRP']
eats         -> MISSING
a            -> FOUND    ['DT', 'IN', 'JJ', 'LS', 'NN', 'NNP']
green        -> FOUND    ['JJ', 'NNP']
salad        -> MISSING
i            -> FOUND    ['NNP', 'PRP']
saw          -> FOUND    ['VBD']
man          -> FOUND    ['NN', 'NNP']
with         -> FOUND    ['IN']
telescope    -> MISSING


In [ ]:
# ============================================================
# SENTENCE-LEVEL BIGRAM / TRIGRAM SCORING
# ============================================================

# Reuse Q3's trained bigram model
Q4_K = 0.5

q4_bigram_freq = bigram_freq
q4_unigram_ctx_freq = unigram_ctx_freq
q4_vocab_size = len(vocab) + 2


def q4_bigram_prob(w1, w2, k=Q4_K):

    return (
        q4_bigram_freq.get(
            (w1, w2),
            0
        ) + k
    ) / (
        q4_unigram_ctx_freq.get(
            w1,
            0
        ) + k * q4_vocab_size
    )


def lm_words(sentence):

    tokens = nltk.word_tokenize(
        sentence.lower()
    )

    return [
        word
        for word in tokens
        if word.isalpha()
    ]


def sentence_bigram_score(sentence):

    words = lm_words(sentence)

    tokens = (
        ["<s>"]
        + words
        + ["</s>"]
    )

    score = 0.0

    for i in range(
        len(tokens) - 1
    ):

        probability = q4_bigram_prob(
            tokens[i],
            tokens[i + 1]
        )

        score += math.log(
            max(
                probability,
                1e-12
            )
        )

    return score


def sentence_trigram_score(sentence):

    words = lm_words(sentence)

    if not words:
        return 0.0

    score = 0.0

    previous_previous = "<s>"
    previous = "<s>"

    # --------------------------------------------------------
    # Reuse Q1's trained English trigram model
    # --------------------------------------------------------

    for word in words:

        probability = lm_en.word_probability(
            previous_previous,
            previous,
            word
        )

        score += math.log(
            max(
                probability,
                1e-12
            )
        )

        previous_previous = previous
        previous = word

    # End-of-sentence probability from Q1 LM
    end_probability = lm_en.word_probability(
        previous_previous,
        previous,
        "</s>"
    )

    score += math.log(
        max(
            end_probability,
            1e-12
        )
    )

    return score


def sentence_perplexity(
    log_probability,
    token_count
):

    if token_count <= 0:
        return float("inf")

    return math.exp(
        -log_probability / token_count
    )


# ------------------------------------------------------------
# Test the shared language models
# ------------------------------------------------------------

example = "the cat sat on the mat"

b_score = sentence_bigram_score(
    example
)

t_score = sentence_trigram_score(
    example
)

print(
    "Sentence:",
    example
)

print(
    "Bigram log probability:",
    b_score
)

print(
    "Trigram log probability:",
    t_score
)

print(
    "Bigram perplexity:",
    sentence_perplexity(
        b_score,
        len(lm_words(example)) + 1
    )
)

print(
    "Trigram perplexity:",
    sentence_perplexity(
        t_score,
        len(lm_words(example)) + 1
    )
)

Sentence: the cat sat on the mat
Bigram log probability: -53.66303968383465
Trigram log probability: -43.800093023878645
Bigram perplexity: 2134.843298030635
Trigram perplexity: 521.7330957361476


In [ ]:
# ============================================================
# PART 1 - SIMULATED FAST-TYPING
# ============================================================

def generate_merged_tokens(
    tokens,
    p=MERGE_PROBABILITY
):

    if not tokens:
        return []

    result = []

    i = 0

    while i < len(tokens):

        if (
            i < len(tokens) - 1
            and tokens[i].isalpha()
            and tokens[i + 1].isalpha()
            and random.random() < p
        ):

            result.append(
                tokens[i] + tokens[i + 1]
            )

            i += 2

        else:

            result.append(
                tokens[i]
            )

            i += 1

    return result

In [ ]:
# ============================================================
# SPELLING ALERT
# ============================================================

def spelling_check(token):

    token = token.lower()

    # Punctuation is not a spelling error
    if not token.isalpha():

        return {
            "changed": False,
            "word": token
        }

    if token in vocab:

        return {
            "changed": False,
            "word": token
        }

    # Reuse Q3's trained corrector
    corrected = correct_nonword(
        token,
        method="B"
    )

    return {
        "changed": corrected != token,
        "word": corrected
    }

In [ ]:
# ============================================================
# TEST SPELLING ALERT
# ============================================================

for token in [
    "hav",
    "sentnce",
    "good"
]:

    print(
        token,
        "=>",
        spelling_check(token)
    )

hav => {'changed': True, 'word': 'had'}
sentnce => {'changed': True, 'word': 'sentence'}
good => {'changed': False, 'word': 'good'}


In [ ]:
# ============================================================
# GRAMMAR + REAL-WORD ALERT
# ============================================================

# A real-word correction must improve the local bigram score
# by at least this amount before it is reported.
REAL_WORD_MIN_LOG_GAIN = 2.0

# Grammar-alert thresholds.
BIGRAM_ALERT_THRESHOLD = -10.0
TRIGRAM_ALERT_THRESHOLD = -12.0


def local_bigram_log_score(
    words,
    index
):
    """
    Calculate the local bigram log-probability involving
    the word at the given index.
    """

    if index < 0 or index >= len(words):
        return -1e9

    word = words[index]

    score = 0.0

    # Left context
    if index == 0:
        score += math.log(
            max(
                q4_bigram_prob(
                    "<s>",
                    word
                ),
                1e-12
            )
        )
    else:
        score += math.log(
            max(
                q4_bigram_prob(
                    words[index - 1],
                    word
                ),
                1e-12
            )
        )

    # Right context
    if index == len(words) - 1:
        score += math.log(
            max(
                q4_bigram_prob(
                    word,
                    "</s>"
                ),
                1e-12
            )
        )
    else:
        score += math.log(
            max(
                q4_bigram_prob(
                    word,
                    words[index + 1]
                ),
                1e-12
            )
        )

    return score


def grammar_trigger_check(words):

    window = [
        str(word).lower()
        for word in words
        if str(word).strip()
    ][-GRAMMAR_TRIGGER_N:]

    if len(window) < 2:
        return {
            "grammar_alert": False,
            "real_word_alerts": [],
            "bigram_score": 0.0,
            "trigram_score": 0.0,
            "avg_bigram": 0.0,
            "avg_trigram": 0.0
        }

    sentence = " ".join(window)

    # Shared Q4 language-model scores
    b_score = sentence_bigram_score(sentence)
    t_score = sentence_trigram_score(sentence)

    # Normalize by the number of scored transitions.
    avg_bigram = (
        b_score / max(len(window) + 1, 1)
    )

    avg_trigram = (
        t_score / max(len(window) + 1, 1)
    )

    real_word_alerts = []

    # --------------------------------------------------------
    # Real-word error detection
    # --------------------------------------------------------

    for i, word in enumerate(window):

        if not word.isalpha():
            continue

        if word not in vocab:
            continue

        corrected, changed = correct_realword(
            window,
            i,
            method="B"
        )

        if not changed:
            continue

        if not corrected:
            continue

        corrected = str(corrected).lower()

        if corrected == word:
            continue

        # Score original word in its context.
        original_local_score = local_bigram_log_score(
            window,
            i
        )

        # Score Q3's proposed correction in the same context.
        candidate_window = list(window)
        candidate_window[i] = corrected

        candidate_local_score = local_bigram_log_score(
            candidate_window,
            i
        )

        log_gain = (
            candidate_local_score
            - original_local_score
        )

        # Only flag the correction when the contextual
        # improvement is substantial.
        if log_gain >= REAL_WORD_MIN_LOG_GAIN:
            real_word_alerts.append({
                "index": i,
                "original": word,
                "suggestion": corrected,
                "log_gain": log_gain
            })

    # --------------------------------------------------------
    # Grammar alert
    # --------------------------------------------------------

    grammar_alert = (
        avg_bigram < BIGRAM_ALERT_THRESHOLD
        or
        avg_trigram < TRIGRAM_ALERT_THRESHOLD
    )

    return {
        "grammar_alert": grammar_alert,
        "real_word_alerts": real_word_alerts,
        "bigram_score": b_score,
        "trigram_score": t_score,
        "avg_bigram": avg_bigram,
        "avg_trigram": avg_trigram
    }

In [ ]:
# ============================================================
# SEGMENTATION ALERT
# ============================================================

def segmentation_check(token):

    token = token.lower()

    # Punctuation is not a segmentation candidate
    if not token.isalpha():
        return {
            "changed": False,
            "words": [token],
            "tags": [None]
        }

    # Known normal-length word
    if (
        token in vocab_en
        and len(token) <= Q1_MAX_WORD_LEN
    ):
        return {
            "changed": False,
            "words": [token],
            "tags": hmm_en.tag([token])
        }

    # Reuse Q1 trained English segmentation model
    split_words = lm_en.segment(token)

    # Only report a segmentation alert when
    # a genuine multi-word split is found.
    if (
        len(split_words) > 1
        and all(
            word in vocab_en
            for word in split_words
        )
    ):

        tags = hmm_en.tag(
            split_words
        )

        return {
            "changed": True,
            "words": split_words,
            "tags": tags
        }

    return {
        "changed": False,
        "words": [token],
        "tags": hmm_en.tag([token])
    }

In [ ]:
# ============================================================
# INTEGRATED LIVE PIPELINE
# ============================================================

def process_live_token(
    token,
    accumulated_words
):

    events = []

    start = time.perf_counter()

    # --------------------------------------------------------
    # Normalize incoming token
    # --------------------------------------------------------

    token = str(token).lower()

    words_before = len(
        accumulated_words
    )

    # --------------------------------------------------------
    # STEP 1: SEGMENTATION
    # --------------------------------------------------------

    seg_result = segmentation_check(
        token
    )

    if seg_result["changed"]:

        events.append({
            "type": "SEGMENT-ALERT",
            "message": (
                f"{token} -> "
                + " | ".join(
                    seg_result["words"]
                )
                + " | POS="
                + str(
                    seg_result["tags"]
                )
            )
        })

        segmentation_words = (
            seg_result["words"]
        )

    else:

        segmentation_words = [
            token
        ]

    # --------------------------------------------------------
    # STEP 2: SPELLING
    # --------------------------------------------------------

    final_words = []

    for word in segmentation_words:

        spell_result = spelling_check(
            word
        )

        if spell_result["changed"]:

            events.append({
                "type": "SPELL-ALERT",
                "message": (
                    f"{word} -> "
                    f"{spell_result['word']}"
                )
            })

            final_words.append(
                spell_result["word"]
            )

        else:

            final_words.append(
                spell_result["word"]
            )

    # --------------------------------------------------------
    # Add processed words
    # --------------------------------------------------------

    accumulated_words.extend(
        final_words
    )

    words_after = len(
        accumulated_words
    )

    per_token_latency = (
        time.perf_counter() - start
    ) * 1000

    # --------------------------------------------------------
    # STEP 3: GRAMMAR + REAL-WORD TRIGGER
    # --------------------------------------------------------

    grammar_result = None
    grammar_latency = 0.0

    next_trigger = (
        (
            words_before
            // GRAMMAR_TRIGGER_N
        ) + 1
    ) * GRAMMAR_TRIGGER_N

    crossed_trigger = (
        words_after >= next_trigger
    )

    if crossed_trigger:

        grammar_start = time.perf_counter()

        window = accumulated_words[
            -GRAMMAR_TRIGGER_N:
        ]

        grammar_result = (
            grammar_trigger_check(
                window
            )
        )

        grammar_latency = (
            time.perf_counter()
            - grammar_start
        ) * 1000

        if grammar_result[
            "grammar_alert"
        ]:

            events.append({
                "type": "GRAMMAR-ALERT",
                "message": (
                    "Implausible recent word "
                    "sequence "
                    f"(bigram avg="
                    f"{grammar_result['avg_bigram']:.2f}, "
                    f"trigram avg="
                    f"{grammar_result['avg_trigram']:.2f})"
                )
            })

        for alert in grammar_result[
            "real_word_alerts"
        ]:

            events.append({
                "type": "GRAMMAR-ALERT",
                "message": (
                    "Possible real-word error: "
                    f"{alert['original']} -> "
                    f"{alert['suggestion']} "
                    f"(log gain="
                    f"{alert['log_gain']:.2f})"
                )
            })

    return {
        "words": final_words,
        "events": events,
        "token_latency_ms": per_token_latency,
        "grammar_latency_ms": grammar_latency,
        "grammar_result": grammar_result
    }

In [ ]:
# ============================================================
# RANDOM CONTIGUOUS PASSAGE GENERATOR
# ============================================================

def sample_random_passage(
    min_sentences=5,
    max_sentences=8,
    min_words=5,
    max_words=25
):

    files = gutenberg.fileids()

    random.shuffle(files)

    for file_id in files:

        raw_text = gutenberg.raw(
            file_id
        )

        sentences = nltk.sent_tokenize(
            raw_text
        )

        sentences = [
            s.strip()
            for s in sentences
        ]

        possible_windows = []

        for n in range(
            min_sentences,
            max_sentences + 1
        ):

            for start in range(
                0,
                len(sentences) - n + 1
            ):

                window = sentences[
                    start:start + n
                ]

                valid = all(
                    min_words
                    <= len(s.split())
                    <= max_words
                    for s in window
                )

                if valid:
                    possible_windows.append(
                        window
                    )

        if possible_windows:

            selected = random.choice(
                possible_windows
            )

            return (
                file_id,
                selected
            )

    raise RuntimeError(
        "Could not find a contiguous Gutenberg passage "
        "with 5–8 sentences within the requested length."
    )


file_id, passage_sentences = (
    sample_random_passage()
)

print(
    "Source:",
    file_id
)

print(
    "\nPASSAGE\n-------"
)

for i, s in enumerate(
    passage_sentences,
    start=1
):

    print(
        f"{i}. {s}"
    )

print(
    "\nNumber of sentences:",
    len(passage_sentences)
)

print(
    "Sentence word counts:",
    [
        len(s.split())
        for s in passage_sentences
    ]
)

assert 5 <= len(passage_sentences) <= 8

Source: chesterton-ball.txt

PASSAGE
-------
1. The house seemed to consist of chamber within chamber and created
that impression as of a dream which belongs also to the Arabian
Nights themselves.
2. The innermost room of all was like the inside
of a jewel.
3. The little man who owned it all threw himself on a
heap of scarlet and golden cushions and struck his hands
together.
4. A negro in a white robe and turban appeared suddenly
and silently behind them.
5. "Selim," said the host, "these two gentlemen are staying with me
tonight.

Number of sentences: 5
Sentence word counts: [25, 12, 22, 14, 12]


In [ ]:
# ============================================================
# LIVE TYPING SIMULATION
# ============================================================

def simulate_live_typing(
    sentences,
    delay=0.03
):

    processed_words = []

    token_latencies = []
    grammar_latencies = []

    segmentation_merges = 0
    spelling_corrections = 0

    sentence_records = []

    print("\n" + "=" * 70)
    print("LIVE TYPING SIMULATION")
    print("=" * 70)

    for sentence in sentences:

        original_tokens = nltk.word_tokenize(
            sentence
        )

        typed_tokens = generate_merged_tokens(
            original_tokens,
            p=MERGE_PROBABILITY
        )

        sentence_start_index = len(
            processed_words
        )

        sentence_segmentation_merges = 0
        sentence_spelling_corrections = 0

        for token in typed_tokens:

            result = process_live_token(
                token,
                processed_words
            )

            token_latencies.append(
                result["token_latency_ms"]
            )

            if result["grammar_latency_ms"] > 0:
                grammar_latencies.append(
                    result["grammar_latency_ms"]
                )

            for event in result["events"]:

                print(
                    f"[{event['type']}] "
                    f"{event['message']}"
                )

                if event["type"] == "SEGMENT-ALERT":

                    segmentation_merges += 1
                    sentence_segmentation_merges += 1

                elif event["type"] == "SPELL-ALERT":

                    spelling_corrections += 1
                    sentence_spelling_corrections += 1

            print(
                "Typed:",
                token,
                " | Processed:",
                " ".join(result["words"])
            )

            time.sleep(delay)

        sentence_end_index = len(
            processed_words
        )

        sentence_records.append({
            "original_sentence": sentence,
            "processed_words": processed_words[
                sentence_start_index:sentence_end_index
            ],
            "segmentation_merges": (
                sentence_segmentation_merges
            ),
            "spelling_corrections": (
                sentence_spelling_corrections
            )
        })

    print("\n" + "-" * 70)
    print("LIVE TYPING SUMMARY")
    print("-" * 70)

    print(
        "Processed words        :",
        len(processed_words)
    )

    print(
        "Segmentation merges    :",
        segmentation_merges
    )

    print(
        "Spelling corrections   :",
        spelling_corrections
    )

    print(
        "Token checks           :",
        len(token_latencies)
    )

    print(
        "Grammar triggers       :",
        len(grammar_latencies)
    )

    if token_latencies:
        print(
            "Average token latency  :",
            f"{statistics.mean(token_latencies):.4f} ms"
        )

    if grammar_latencies:
        print(
            "Average grammar latency:",
            f"{statistics.mean(grammar_latencies):.4f} ms"
        )

    return {
        "words": processed_words,
        "token_latencies": token_latencies,
        "grammar_latencies": grammar_latencies,
        "segmentation_merges": segmentation_merges,
        "spelling_corrections": spelling_corrections,
        "sentence_records": sentence_records
    }


live_result = simulate_live_typing(
    passage_sentences
)


LIVE TYPING SIMULATION
Typed: The  | Processed: the
Typed: house  | Processed: house
Typed: seemed  | Processed: seemed
Typed: to  | Processed: to
Typed: consist  | Processed: consist
Typed: of  | Processed: of
Typed: chamber  | Processed: chamber
Typed: within  | Processed: within
Typed: chamber  | Processed: chamber
Typed: and  | Processed: and
Typed: created  | Processed: created
[SEGMENT-ALERT] thatimpression -> that | impression | POS=['DET', 'NOUN']
Typed: thatimpression  | Processed: that impression
Typed: as  | Processed: as
[SEGMENT-ALERT] ofa -> of | a | POS=['ADP', 'DET']
Typed: ofa  | Processed: of a
Typed: dream  | Processed: dream
Typed: which  | Processed: which
Typed: belongs  | Processed: belongs
Typed: also  | Processed: also
Typed: to  | Processed: to
Typed: the  | Processed: the
Typed: Arabian  | Processed: arabian
[GRAMMAR-ALERT] Possible real-word error: dream -> dreams (log gain=3.22)
Typed: Nights  | Processed: nights
Typed: themselves  | Processed: themselves


In [ ]:
# ============================================================
# LATENCY REPORT
# ============================================================

token_latencies = live_result[
    "token_latencies"
]

grammar_latencies = [
    x
    for x in live_result["grammar_latencies"]
    if x > 0
]

avg_token_latency = (
    statistics.mean(token_latencies)
    if token_latencies
    else 0
)

avg_grammar_latency = (
    statistics.mean(grammar_latencies)
    if grammar_latencies
    else 0
)

print("=" * 60)
print("LATENCY REPORT")
print("=" * 60)

print(
    f"Average segmentation + spelling latency: "
    f"{avg_token_latency:.3f} ms"
)

print(
    f"Average grammar trigger latency: "
    f"{avg_grammar_latency:.3f} ms"
)

print(
    f"Segmentation merges resolved: "
    f"{live_result['segmentation_merges']}"
)

LATENCY REPORT
Average segmentation + spelling latency: 0.119 ms
Average grammar trigger latency: 0.894 ms
Segmentation merges resolved: 7


In [ ]:
# ============================================================
# PART 4 - FINAL PASSAGE ANALYSIS
# ============================================================

def analyze_sentence(sentence):
    """
    Analyze one final corrected sentence using:
      1. Q1 English POS + tag reconciliation
      2. Q4 PCFG / CKY parser
      3. Q3 bigram model
      4. Q1 trigram model
    """

    # --------------------------------------------------------
    # Tokenize
    # --------------------------------------------------------

    raw_words = nltk.word_tokenize(
        sentence.lower()
    )

    parse_words = [
        word
        for word in raw_words
        if word.isalpha()
    ]

    if not parse_words:
        return {
            "sentence": sentence,
            "pcfg_result": "UNPARSEABLE",
            "pcfg_score": -1e9,
            "bigram_score": -1e9,
            "trigram_score": -1e9,
            "chosen_method": "Bigram",
            "final_verdict": "Possibly ungrammatical"
        }

    # --------------------------------------------------------
    # Q1 POS -> Penn Treebank tag reconciliation
    # --------------------------------------------------------

    q1_tags, ptb_tags = reconcile_q1_tags(
        parse_words
    )

    # --------------------------------------------------------
    # PCFG / CKY
    # --------------------------------------------------------

    tree, pcfg_score = cky_parse(
        parse_words,
        q1_ptb_tags=ptb_tags
    )

    if tree is None:
        pcfg_result = "UNPARSEABLE"
    else:
        pcfg_result = "PARSED"

    # --------------------------------------------------------
    # N-gram scores
    # --------------------------------------------------------

    bigram_score = sentence_bigram_score(
        " ".join(parse_words)
    )

    trigram_score = sentence_trigram_score(
    " ".join(parse_words)
    )

    token_count = max(
        len(parse_words) + 1,
        1
    )

    trigram_avg = (
        trigram_score / token_count
    )

    # --------------------------------------------------------
    # Documented decision rule
    #
    # 1. Prefer PCFG if the sentence parses.
    # 2. Otherwise use trigram if its normalized
    #    score is acceptable.
    # 3. Otherwise use bigram.
    # --------------------------------------------------------

    if pcfg_result == "PARSED":

        chosen_method = "PCFG"
        verdict = "Likely grammatical"

    elif trigram_avg >= TRIGRAM_ALERT_THRESHOLD:

        chosen_method = "Trigram"
        verdict = "Likely grammatical"

    else:

        chosen_method = "Bigram"
        verdict = "Possibly ungrammatical"

    return {
        "sentence": sentence,
        "pcfg_result": pcfg_result,
        "pcfg_score": pcfg_score,
        "bigram_score": bigram_score,
        "trigram_score": trigram_score,
        "chosen_method": chosen_method,
        "final_verdict": verdict
    }

In [ ]:
# ============================================================
# FINAL PASSAGE ANALYSIS WITH CORRECTION COUNTS
# ============================================================

def analyze_passage_with_counters(
    sentences
):

    rows = []

    for sentence in sentences:

        original_words = sentence.split()

        # Simulate missed spaces
        typed_tokens = generate_merged_tokens(
            original_words,
            p=MERGE_PROBABILITY
        )

        processed_words = []

        merge_count = 0
        spelling_count = 0

        # ----------------------------------------------------
        # Segmentation + spelling
        # ----------------------------------------------------

        for token in typed_tokens:

            seg_result = segmentation_check(
                token
            )

            if seg_result["changed"]:
                merge_count += 1

            words_after_seg = (
                seg_result["words"]
            )

            for word in words_after_seg:

                spell_result = spelling_check(
                    word
                )

                if spell_result["changed"]:
                    spelling_count += 1

                processed_words.append(
                    spell_result["word"]
                )

        # ----------------------------------------------------
        # Final corrected sentence
        # ----------------------------------------------------

        final_sentence = " ".join(
            processed_words
        )

        result = analyze_sentence(
            final_sentence
        )

        result["segmentation_merges"] = (
            merge_count
        )

        result["spelling_corrections"] = (
            spelling_count
        )

        rows.append(result)

    return pd.DataFrame(rows)


# Analyze the complete 5–8 sentence passage
final_analysis_df = (
    analyze_passage_with_counters(
        passage_sentences
    )
)

display(final_analysis_df)

print(
    "\nNumber of analyzed sentences:",
    len(final_analysis_df)
)

assert 5 <= len(final_analysis_df) <= 8

,sentence,pcfg_result,pcfg_score,bigram_score,trigram_score,chosen_method,final_verdict,segmentation_merges,spelling_corrections
0,the house seemed to consist of chamber within ...,PARSED,-221.714923,-219.072504,-176.825680,PCFG,Likely grammatical,2,0
1,the innermost room of all was like the inside ...,PARSED,-114.025873,-98.028637,-79.994507,PCFG,Likely grammatical,0,0
2,the little man who owned it all threw himself ...,PARSED,-240.646949,-190.136659,-147.602154,PCFG,Likely grammatical,1,0
3,a negro in a white robe and turban appeared su...,PARSED,-153.974481,-124.817566,-109.804301,PCFG,Likely grammatical,0,0
4,"""selim,"" said the host, ""these two gentlemen a...",PARSED,-136.684146,-119.906401,-116.653342,PCFG,Likely grammatical,0,0



Number of analyzed sentences: 5


In [ ]:
# ============================================================
# INTERACTION ANALYSIS
# ============================================================

print("=" * 70)
print("Q4 INTERACTION ANALYSIS")
print("=" * 70)

total_sentences = len(
    final_analysis_df
)

seg_corrections = (
    final_analysis_df[
        "segmentation_merges"
    ].sum()
)

spell_corrections = (
    final_analysis_df[
        "spelling_corrections"
    ].sum()
)

pcfg_parsed = (
    final_analysis_df[
        "pcfg_result"
    ] == "PARSED"
).sum()

print(
    f"Total sentences analyzed       : "
    f"{total_sentences}"
)

print(
    f"Segmentation merges corrected  : "
    f"{seg_corrections}"
)

print(
    f"Spelling corrections applied   : "
    f"{spell_corrections}"
)

print(
    f"Sentences successfully parsed  : "
    f"{pcfg_parsed}/{total_sentences}"
)

print(
    "\nInterpretation:"
)

print(
    "Segmentation and spelling corrections are performed "
    "before the PCFG analysis, so an upstream correction "
    "can change whether a sentence is parseable and can "
    "also change the n-gram method selected at the end."
)

Q4 INTERACTION ANALYSIS
Total sentences analyzed       : 5
Segmentation merges corrected  : 3
Spelling corrections applied   : 0
Sentences successfully parsed  : 5/5

Interpretation:
Segmentation and spelling corrections are performed before the PCFG analysis, so an upstream correction can change whether a sentence is parseable and can also change the n-gram method selected at the end.


In [ ]:
# ============================================================
# PART 5 - SPEED DEMON BENCHMARK
# ============================================================

# ------------------------------------------------------------
# Create a simple synthetic non-word corruption
# ------------------------------------------------------------

def introduce_error(word):

    word = str(word).lower()

    if len(word) < 2:
        return word + "x"

    # Delete one character from the middle.
    # This creates a non-word while remaining realistic
    # for spelling-correction benchmarking.
    pos = random.randint(
        0,
        len(word) - 1
    )

    corrupted = (
        word[:pos]
        + word[pos + 1:]
    )

    # Make sure the result is actually different.
    if corrupted == word:
        corrupted = word + "x"

    return corrupted


# ------------------------------------------------------------
# Build exactly 1,000 corrupted non-word tokens
# ------------------------------------------------------------

vocab_list_q4 = [
    word
    for word in vocab
    if str(word).isalpha()
]

benchmark_words = []

while len(benchmark_words) < 1000:

    original = random.choice(
        vocab_list_q4
    )

    corrupted = introduce_error(
        original
    )

    if corrupted not in vocab:

        benchmark_words.append(
            corrupted
        )

assert len(benchmark_words) == 1000

print(
    "Verified benchmark size:",
    len(benchmark_words)
)

Verified benchmark size: 1000


In [ ]:
# ============================================================
# REQUIRED TEST SENTENCES
# ============================================================

required_tests = [
    "The cat sat on the mat.",
    "She eats a green salad.",
    "I saw the man with a telescope.",
    "I hav a good feeling about this.",
    "This is a test sentnce.",
    "I would like to sea the world.",
    "Please meat me at the station."
]

for sentence in required_tests:

    print("\n" + "=" * 70)
    print("INPUT:", sentence)

    words = sentence.lower().split()

    result = analyze_sentence(
        " ".join(words)
    )

    print(
        "PCFG:",
        result["pcfg_result"]
    )

    print(
        "Bigram:",
        result["bigram_score"]
    )

    print(
        "Trigram:",
        result["trigram_score"]
    )

    print(
        "Chosen:",
        result["chosen_method"]
    )

    print(
        "Verdict:",
        result["final_verdict"]
    )


INPUT: The cat sat on the mat.
PCFG: PARSED
Bigram: -53.66303968383465
Trigram: -43.800093023878645
Chosen: PCFG
Verdict: Likely grammatical

INPUT: She eats a green salad.
PCFG: PARSED
Bigram: -55.909853672041145
Trigram: -50.30500967659583
Chosen: PCFG
Verdict: Likely grammatical

INPUT: I saw the man with a telescope.
PCFG: PARSED
Bigram: -51.606500036186816
Trigram: -37.032685322618285
Chosen: PCFG
Verdict: Likely grammatical

INPUT: I hav a good feeling about this.
PCFG: PARSED
Bigram: -59.04678594063677
Trigram: -60.09816095906149
Chosen: PCFG
Verdict: Likely grammatical

INPUT: This is a test sentnce.
PCFG: PARSED
Bigram: -41.76580256788056
Trigram: -50.07720754454767
Chosen: PCFG
Verdict: Likely grammatical

INPUT: I would like to sea the world.
PCFG: PARSED
Bigram: -50.212493298084546
Trigram: -34.01113223915203
Chosen: PCFG
Verdict: Likely grammatical

INPUT: Please meat me at the station.
PCFG: PARSED
Bigram: -55.79069410661702
Trigram: -45.69381067678786
Chosen: PCFG
Verdi

In [ ]:
# ============================================================
# SAVE Q4 MODEL BUNDLE
# ============================================================

import cloudpickle

q4_bundle = {

    # --------------------------------------------------------
    # Q1
    # --------------------------------------------------------

    "lm_en": lm_en,
    "hmm_en": hmm_en,
    "vocab_en": vocab_en,

    # --------------------------------------------------------
    # Q3
    # --------------------------------------------------------

    "vocab": vocab,
    "unigram_freq": unigram_freq,
    "bigram_freq": bigram_freq,
    "unigram_ctx_freq": unigram_ctx_freq,
    "symspell_index": symspell_index,

    "correct_nonword": correct_nonword,
    "correct_realword": correct_realword,

    # --------------------------------------------------------
    # Q4 PCFG
    # --------------------------------------------------------

    "pcfg_grammar": pcfg_grammar,
    "binary_rules": dict(binary_rules),
    "unary_rules": dict(unary_rules),
    "lexical_rules": dict(lexical_rules),

    # --------------------------------------------------------
    # Q4 parser/helper functions
    # --------------------------------------------------------

    "cky_parse": cky_parse,
    "reconcile_q1_tags": reconcile_q1_tags,
    "analyze_sentence": analyze_sentence,

    # --------------------------------------------------------
    # Q4 configuration
    # --------------------------------------------------------

    "MERGE_PROBABILITY": MERGE_PROBABILITY,
    "GRAMMAR_TRIGGER_N": GRAMMAR_TRIGGER_N,
    "Q4_K": Q4_K
}


with open(
    "q4_model_bundle.pkl",
    "wb"
) as f:

    cloudpickle.dump(
        q4_bundle,
        f
    )


print(
    "Saved Q4 model bundle successfully."
)

print(
    "Bundle contains Q1 + Q3 + Q4 models and parser functions."
)

print(
    "q4_model_bundle.pkl"
)

Saved Q4 model bundle successfully.
Bundle contains Q1 + Q3 + Q4 models and parser functions.
q4_model_bundle.pkl


In [ ]:
%%writefile q4_app.py

import os
import re
import time
import math
import statistics

import cloudpickle
import pandas as pd
import streamlit as st
import nltk

from collections import defaultdict


# ============================================================
# SETUP
# ============================================================

nltk.download(
    "punkt",
    quiet=True
)

nltk.download(
    "punkt_tab",
    quiet=True
)


# ============================================================
# LOAD Q4 BUNDLE
# ============================================================

BASE_DIR = os.path.dirname(
    os.path.abspath(__file__)
)

MODEL_PATH = os.path.join(
    BASE_DIR,
    "q4_model_bundle.pkl"
)

with open(
    MODEL_PATH,
    "rb"
) as f:

    model = cloudpickle.load(f)


# ------------------------------------------------------------
# Q1
# ------------------------------------------------------------

lm_en = model["lm_en"]
hmm_en = model["hmm_en"]
vocab_en = model["vocab_en"]


# ------------------------------------------------------------
# Q3
# ------------------------------------------------------------

vocab = model["vocab"]
unigram_freq = model["unigram_freq"]
bigram_freq = model["bigram_freq"]
unigram_ctx_freq = model[
    "unigram_ctx_freq"
]

symspell_index = model[
    "symspell_index"
]

correct_nonword = model[
    "correct_nonword"
]

correct_realword = model[
    "correct_realword"
]


# ------------------------------------------------------------
# Q4 PCFG
# ------------------------------------------------------------

pcfg_grammar = model[
    "pcfg_grammar"
]

binary_rules = defaultdict(
    list,
    model["binary_rules"]
)

unary_rules = defaultdict(
    list,
    model["unary_rules"]
)

lexical_rules = defaultdict(
    list,
    model["lexical_rules"]
)
# ------------------------------------------------------------
# Load Q4 parser/helper functions
# ------------------------------------------------------------

cky_parse = model["cky_parse"]
reconcile_q1_tags = model["reconcile_q1_tags"]
analyze_sentence = model["analyze_sentence"]


# ------------------------------------------------------------
# Q4 settings
# ------------------------------------------------------------

MERGE_PROBABILITY = model[
    "MERGE_PROBABILITY"
]

GRAMMAR_TRIGGER_N = model[
    "GRAMMAR_TRIGGER_N"
]

Q4_K = model[
    "Q4_K"
]

REAL_WORD_MIN_LOG_GAIN = 2.0

TRIGRAM_ALERT_THRESHOLD = -12.0

BIGRAM_ALERT_THRESHOLD = -10.0


# ============================================================
# Q1 SEGMENTATION
# ============================================================

def segmentation_check(token):

    token = str(token).lower()

    if not token:
        return {
            "changed": False,
            "words": [],
            "tags": []
        }

    if not token.isalpha():

        return {
            "changed": False,
            "words": [token],
            "tags": [None]
        }

    # Reuse the Q1 English segmentation model
    split_words = lm_en.segment(
        token
    )

    # If the result is a true multi-word split,
    # report a segmentation alert.
    if (
        len(split_words) > 1
        and all(
            word in vocab_en
            for word in split_words
        )
    ):

        tags = hmm_en.tag(
            split_words
        )

        return {
            "changed": True,
            "words": split_words,
            "tags": tags
        }

    return {
        "changed": False,
        "words": [token],
        "tags": hmm_en.tag(
            [token]
        )
    }


# ============================================================
# Q3 SPELLING
# ============================================================

def spelling_check(token):

    token = str(token).lower()

    if not token.isalpha():

        return {
            "changed": False,
            "word": token
        }

    # Q3 non-word correction
    if token not in vocab:

        corrected = correct_nonword(
            token,
            method="B"
        )

        return {
            "changed": corrected != token,
            "word": corrected
        }

    return {
        "changed": False,
        "word": token
    }


# ============================================================
# Q4 BIGRAM / TRIGRAM
# ============================================================

V = len(vocab) + 2


def q4_bigram_prob(w1, w2):

    return (
        bigram_freq.get(
            (w1, w2),
            0
        ) + Q4_K
    ) / (
        unigram_ctx_freq.get(
            w1,
            0
        )
        + Q4_K * V
    )


def sentence_bigram_score(words):

    tokens = (
        ["<s>"]
        + [
            str(word).lower()
            for word in words
            if str(word).strip()
        ]
        + ["</s>"]
    )

    score = 0.0

    for i in range(
        len(tokens) - 1
    ):

        p = q4_bigram_prob(
            tokens[i],
            tokens[i + 1]
        )

        score += math.log(
            max(p, 1e-12)
        )

    return score


def sentence_trigram_score(words):

    clean_words = [
        str(word).lower()
        for word in words
        if str(word).strip()
    ]

    if not clean_words:
        return 0.0

    score = 0.0

    previous_previous = "<s>"
    previous = "<s>"

    # Reuse Q1's trained English trigram LM
    for word in clean_words:

        p = lm_en.word_probability(
            previous_previous,
            previous,
            word
        )

        score += math.log(
            max(p, 1e-12)
        )

        previous_previous = previous
        previous = word

    end_p = lm_en.word_probability(
        previous_previous,
        previous,
        "</s>"
    )

    score += math.log(
        max(end_p, 1e-12)
    )

    return score


# ============================================================
# REAL-WORD CHECK
# ============================================================

def local_bigram_log_score(
    words,
    index
):

    if (
        index < 0
        or index >= len(words)
    ):
        return -1e9

    word = words[index]

    previous = (
        words[index - 1]
        if index > 0
        else "<s>"
    )

    following = (
        words[index + 1]
        if index < len(words) - 1
        else "</s>"
    )

    score = 0.0

    score += math.log(
        max(
            q4_bigram_prob(
                previous,
                word
            ),
            1e-12
        )
    )

    score += math.log(
        max(
            q4_bigram_prob(
                word,
                following
            ),
            1e-12
        )
    )

    return score


# ============================================================
# GRAMMAR TRIGGER
# ============================================================

def grammar_trigger_check(
    words
):

    window = [
        str(word).lower()
        for word in words
        if str(word).strip()
    ][-GRAMMAR_TRIGGER_N:]

    if len(window) < 2:

        return {
            "grammar_alert": False,
            "real_word_alerts": [],
            "bigram_score": 0.0,
            "trigram_score": 0.0
        }

    b_score = sentence_bigram_score(
        window
    )

    t_score = sentence_trigram_score(
        window
    )

    real_word_alerts = []

    # --------------------------------------------------------
    # Real-word error detection
    # --------------------------------------------------------

    for i, word in enumerate(window):

        if not word.isalpha():
            continue

        if word not in vocab:
            continue

        try:

            corrected, changed = (
                correct_realword(
                    window,
                    i,
                    method="B"
                )
            )

        except Exception:
            continue

        if not changed:
            continue

        if corrected == word:
            continue

        candidate_window = list(
            window
        )

        candidate_window[i] = corrected

        original_score = (
            local_bigram_log_score(
                window,
                i
            )
        )

        candidate_score = (
            local_bigram_log_score(
                candidate_window,
                i
            )
        )

        gain = (
            candidate_score
            - original_score
        )

        if gain >= REAL_WORD_MIN_LOG_GAIN:

            real_word_alerts.append({
                "index": i,
                "original": word,
                "suggestion": corrected,
                "log_gain": gain
            })

    avg_bigram = (
        b_score
        / max(len(window) + 1, 1)
    )

    avg_trigram = (
        t_score
        / max(len(window) + 1, 1)
    )

    grammar_alert = (
        avg_bigram < BIGRAM_ALERT_THRESHOLD
        or
        avg_trigram < TRIGRAM_ALERT_THRESHOLD
    )

    return {
        "grammar_alert": grammar_alert,
        "real_word_alerts": real_word_alerts,
        "bigram_score": b_score,
        "trigram_score": t_score
    }


# ============================================================
# PUNCTUATION
# ============================================================

def split_trailing_punct(
    token
):

    match = re.match(
        r"^([a-z]+)([^a-z]*)$",
        token.lower()
    )

    if match:

        return (
            match.group(1),
            match.group(2)
        )

    return token, ""


# ============================================================
# STREAMLIT UI
# ============================================================

st.set_page_config(
    page_title="Q4 Background Editor",
    page_icon="✍️",
    layout="wide"
)

st.title(
    "✍️ Integrated Background Editor"
)

st.caption(
    "Question 1 segmentation + "
    "Question 3 spelling correction + "
    "Q4 grammar checking"
)


# ============================================================
# LIVE MODE
# ============================================================

mode = st.radio(
    "Mode",
    [
        "Live Typing",
        "Simulated Passage"
    ]
)


# ============================================================
# LIVE TYPING MODE
# ============================================================

if mode == "Live Typing":

    text = st.text_area(
        "Type your text:",
        height=220,
        placeholder="Start typing..."
    )

    if text:

        raw_tokens = text.split()

        processed = []
        alerts = []

        token_latencies = []

        segmentation_merges = 0
        spelling_corrections = 0

        # ----------------------------------------------------
        # Process incoming tokens
        # ----------------------------------------------------

        for raw_token in raw_tokens:

            start = time.perf_counter()

            core, punct = (
                split_trailing_punct(
                    raw_token
                )
            )

            # -----------------------------------------------
            # Segmentation
            # -----------------------------------------------

            seg_result = (
                segmentation_check(core)
            )

            segment_words = (
                seg_result["words"]
            )

            if seg_result["changed"]:

                segmentation_merges += 1

                alerts.append(
                    "[SEGMENT-ALERT] "
                    f"{raw_token} -> "
                    f"{' | '.join(segment_words)}"
                )

            # -----------------------------------------------
            # Spelling
            # -----------------------------------------------

            corrected_words = []

            for word in segment_words:

                spelling_result = (
                    spelling_check(word)
                )

                if spelling_result["changed"]:

                    spelling_corrections += 1

                    alerts.append(
                        "[SPELL-ALERT] "
                        f"{word} -> "
                        f"{spelling_result['word']}"
                    )

                corrected_words.append(
                    spelling_result["word"]
                )

            # Add punctuation back
            if punct:

                corrected_words[-1] = (
                    corrected_words[-1]
                    + punct
                )

            processed.extend(
                corrected_words
            )

            elapsed_ms = (
                time.perf_counter()
                - start
            ) * 1000

            token_latencies.append(
                elapsed_ms
            )

            # -----------------------------------------------
            # Grammar every N words
            # -----------------------------------------------

            clean_processed = [
                word
                for word in processed
                if word.isalpha()
            ]

            if (
                len(clean_processed) >= GRAMMAR_TRIGGER_N
                and
                len(clean_processed)
                % GRAMMAR_TRIGGER_N
                == 0
            ):

                result = grammar_trigger_check(
                    clean_processed
                )

                if result["grammar_alert"]:

                    alerts.append(
                        "[GRAMMAR-ALERT] "
                        "The recent word sequence "
                        "looks statistically implausible."
                    )

                for alert in result[
                    "real_word_alerts"
                ]:

                    alerts.append(
                        "[GRAMMAR-ALERT] "
                        "Possible real-word error: "
                        f"{alert['original']} -> "
                        f"{alert['suggestion']}"
                    )

        # ----------------------------------------------------
        # Display processed text
        # ----------------------------------------------------

        st.subheader(
            "Processed Text"
        )

        st.write(
            " ".join(processed)
        )

        # ----------------------------------------------------
        # Alerts
        # ----------------------------------------------------

        st.subheader(
            "Live Alerts"
        )

        if alerts:

            for alert in alerts:

                if alert.startswith(
                    "[SEGMENT-ALERT]"
                ):

                    st.info(alert)

                elif alert.startswith(
                    "[SPELL-ALERT]"
                ):

                    st.warning(alert)

                else:

                    st.error(alert)

        else:

            st.success(
                "No alerts detected."
            )

        # ----------------------------------------------------
        # Latency
        # ----------------------------------------------------

        if token_latencies:

            st.subheader(
                "Latency"
            )

            st.metric(
                "Average segmentation + spelling latency",
                f"{statistics.mean(token_latencies):.3f} ms"
            )

        st.metric(
            "Segmentation merges resolved",
            segmentation_merges
        )

        st.metric(
            "Spelling corrections",
            spelling_corrections
        )

        # ----------------------------------------------------
        # Final sentence analysis
        # ----------------------------------------------------

        st.subheader(
            "Final Sentence Analysis"
        )

        corrected_text = " ".join(
            processed
        )

        sentences = nltk.sent_tokenize(
            corrected_text
        )

        rows = []

        for sentence in sentences:

            words = [
                word
                for word in nltk.word_tokenize(
                    sentence.lower()
                )
                if word.isalpha()
            ]

            if not words:
                continue


            alpha_q1_tags = hmm_en.tag(
                words
            )

            ptb_tags = [
                {
                    "NOUN": "NN",
                    "PROPN": "NNP",
                    "VERB": "VB",
                    "AUX": "VB",
                    "ADJ": "JJ",
                    "ADV": "RB",
                    "PRON": "PRP",
                    "DET": "DT",
                    "ADP": "IN",
                    "NUM": "CD",
                    "CCONJ": "CC",
                    "SCONJ": "IN",
                    "PART": "RP",
                    "INTJ": "UH",
                    "SYM": "SYM",
                    "X": "FW"
                }.get(tag)
                for tag in alpha_q1_tags
            ]

            tree, pcfg_score = (
                None,
                -1e9
            )

            try:

                tree, pcfg_score = cky_parse(
                    words,
                    q1_ptb_tags=ptb_tags
                )

            except Exception:

                pass

            pcfg_result = (
                "PARSED"
                if tree is not None
                else "UNPARSEABLE"
            )

            b_score = sentence_bigram_score(
                words
            )

            t_score = sentence_trigram_score(
                words
            )

            if pcfg_result == "PARSED":

                chosen_method = "PCFG"
                verdict = "Likely grammatical"

            elif (
                t_score
                / max(len(words) + 1, 1)
                >= TRIGRAM_ALERT_THRESHOLD
            ):

                chosen_method = "Trigram"
                verdict = "Likely grammatical"

            else:

                chosen_method = "Bigram"
                verdict = "Possibly ungrammatical"

            rows.append({
                "sentence": sentence,
                "pcfg_result": pcfg_result,
                "pcfg_score": pcfg_score,
                "bigram_score": b_score,
                "trigram_score": t_score,
                "chosen_method": chosen_method,
                "final_verdict": verdict,
                "segmentation_merges": 0,
                "spelling_corrections": 0
            })

        st.dataframe(
            pd.DataFrame(rows),
            use_container_width=True
        )


# ============================================================
# SIMULATED PASSAGE MODE
# ============================================================

else:

    st.subheader(
        "Simulated Live Passage"
    )

    if "simulated_started" not in st.session_state:
        st.session_state.simulated_started = False

    if "simulated_result" not in st.session_state:
        st.session_state.simulated_result = []

    uploaded_text = st.text_area(
        "Optional passage:",
        height=180
    )

    if st.button(
        "Process Passage"
    ):

        tokens = (
            uploaded_text.split()
            if uploaded_text.strip()
            else []
        )

        processed = []
        alerts = []

        for token in tokens:

            core, punct = (
                split_trailing_punct(
                    token
                )
            )

            seg = segmentation_check(
                core
            )

            if seg["changed"]:

                alerts.append(
                    "[SEGMENT-ALERT] "
                    f"{token} -> "
                    f"{' | '.join(seg['words'])}"
                )

            for word in seg["words"]:

                spell = spelling_check(
                    word
                )

                if spell["changed"]:

                    alerts.append(
                        "[SPELL-ALERT] "
                        f"{word} -> "
                        f"{spell['word']}"
                    )

                final_word = spell["word"]

                if punct and word == seg["words"][-1]:

                    final_word += punct

                processed.append(
                    final_word
                )

        st.session_state.simulated_result = (
            processed
        )

        st.session_state.simulated_started = True

        st.session_state.simulated_alerts = (
            alerts
        )

    if st.session_state.simulated_started:

        st.subheader(
            "Processed Passage"
        )

        st.write(
            " ".join(
                st.session_state.simulated_result
            )
        )

        st.subheader(
            "Alerts"
        )

        for alert in st.session_state.get(
            "simulated_alerts",
            []
        ):

            st.warning(alert)

        st.success(
            "Passage processed successfully."
        )

Overwriting q4_app.py


In [ ]:
!pip install -q streamlit

In [ ]:
# ============================================================
# START Q4 STREAMLIT APP - COLAB
# ============================================================

!pkill -f "streamlit run q4_app.py" || true

!python -m streamlit run q4_app.py \
    --server.address 0.0.0.0 \
    --server.port 8501 \
    --server.headless true \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    > streamlit.log 2>&1 &

import time
time.sleep(5)

print("=" * 60)
print("STREAMLIT LOG")
print("=" * 60)

!cat streamlit.log

^C
STREAMLIT LOG




In [159]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>